# Step 4 — Build the Transformer Decoder Model
**Medical Chatbot | GPT-style Autoregressive Model**

| Parameter | Value |
|-----------|-------|
| Architecture | Transformer Decoder (GPT-style) |
| d_model | 256 |
| n_heads | 4 |
| n_layers | 6 |
| ffn_dim | 1024 |
| vocab_size | 8000 |
| max_seq_len | 512 |


## Cell 1 — Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import json
import os

print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')

PyTorch version : 2.5.1+cu121
CUDA available  : True
GPU             : NVIDIA GeForce RTX 4060 Ti
VRAM            : 15.6 GB


## Cell 2 — Load Config

In [2]:
with open('../tokenizer/config.json', 'r') as f:
    config = json.load(f)

VOCAB_SIZE   = config['vocab_size']    # 8000
MAX_SEQ_LEN  = config['max_seq_len']   # 512
D_MODEL      = config['d_model']       # 256
N_HEADS      = config['n_heads']       # 8
N_LAYERS     = config['n_layers']      # 6
FFN_DIM      = config['ffn_dim']       # 1024
PAD_ID       = config['pad_id']        # 0
EOS_ID       = config['eos_id']        # 4

print('=== Config Loaded ===')
for k, v in config.items():
    print(f'  {k:<15} : {v}')

=== Config Loaded ===
  model_path      : ../tokenizer/medical_bpe.model
  vocab_size      : 8000
  max_seq_len     : 512
  batch_size      : 64
  d_model         : 256
  n_heads         : 4
  n_layers        : 6
  ffn_dim         : 1024
  pad_id          : 0
  unk_id          : 1
  eos_id          : 4
  patient_id      : 5
  doctor_id       : 6


## Cell 3 — Token + Positional Embedding

**Token Embedding** : Converts each token ID → a vector of size d_model (256)

**Positional Embedding** : Adds position information so the model knows word order

Both are **learned** during training (not fixed)

In [3]:
class Embeddings(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len, pad_id, dropout=0.1):
        super().__init__()
        # Token embedding: each token ID → 256-dim vector
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        # Positional embedding: each position 0..511 → 256-dim vector
        self.pos_emb   = nn.Embedding(max_seq_len, d_model)
        self.dropout   = nn.Dropout(dropout)
        self.d_model   = d_model

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        seq_len   = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)  # (1, seq_len)
        
        tok = self.token_emb(x)   # (batch, seq_len, 256)
        pos = self.pos_emb(positions)  # (1, seq_len, 256)
        
        # Scale token embeddings (standard transformer trick)
        out = tok * math.sqrt(self.d_model) + pos
        return self.dropout(out)

print('Embeddings class defined OK')

Embeddings class defined OK


## Cell 4 — Masked Multi-Head Self Attention

**Why Masked?** This is a decoder — when predicting token at position 5, the model can only see tokens 0-4. It cannot look at future tokens. The mask blocks future positions.

**Multi-Head** : 4 attention heads, each looks at different parts of the sequence. Each head has dimension 256/4 = 64.

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, 'd_model must be divisible by n_heads'
        
        self.d_model  = d_model            # 256
        self.n_heads  = n_heads            # 4
        self.d_head   = d_model // n_heads # 64 per head
        
        # Q, K, V projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Output projection
        self.W_o    = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, causal_mask=None, pad_mask=None):
        # x shape: (batch, seq_len, d_model)
        B, T, _ = x.shape
        
        # Project to Q, K, V
        Q = self.W_q(x)  # (B, T, d_model)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Split into n_heads
        # (B, T, d_model) → (B, n_heads, T, d_head)
        Q = Q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        K = K.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        V = V.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        
        # Scaled dot-product attention
        scale  = math.sqrt(self.d_head)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / scale  # (B, n_heads, T, T)
        
        # Apply causal mask (block future tokens)
        if causal_mask is not None:
            scores = scores.masked_fill(causal_mask == 0, float('-inf'))
        
        # Apply padding mask (ignore pad tokens)
        if pad_mask is not None:
            # pad_mask: (B, T) → (B, 1, 1, T)
            pad_mask = pad_mask.unsqueeze(1).unsqueeze(2)
            scores   = scores.masked_fill(pad_mask == 0, float('-inf'))
        
        # Softmax + dropout
        attn   = F.softmax(scores, dim=-1)
        attn   = torch.nan_to_num(attn, nan=0.0)  # handle -inf rows
        attn   = self.dropout(attn)
        
        # Weighted sum of values
        out = torch.matmul(attn, V)  # (B, n_heads, T, d_head)
        
        # Merge heads back
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)  # (B, T, d_model)
        out = self.W_o(out)
        
        return out

print('MultiHeadAttention class defined OK')

MultiHeadAttention class defined OK


## Cell 5 — Feed Forward Network

Applied after attention in each layer. Expands 256 → 1024 → 256. This is where the model learns to transform features.

In [5]:
class FeedForward(nn.Module):
    def __init__(self, d_model, ffn_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, ffn_dim),   # 256 → 1024
            nn.GELU(),                      # smooth activation
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),   # 1024 → 256
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

print('FeedForward class defined OK')

FeedForward class defined OK


## Cell 6 — Decoder Layer (One Block)

Each layer = Attention + FFN, both wrapped with Residual Connection + LayerNorm

```
Input
  │
  ├─→ LayerNorm → MultiHeadAttention → + (residual)
  │                                     │
  └─────────────────────────────────────┘
                                         │
  ┌──────────────────────────────────────┘
  │
  ├─→ LayerNorm → FeedForward → + (residual)
  │                              │
  └──────────────────────────────┘
                                  │
                                Output
```

In [6]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.attn    = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn     = FeedForward(d_model, ffn_dim, dropout)
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, causal_mask=None, pad_mask=None):
        # Pre-norm attention + residual
        attn_out = self.attn(self.norm1(x), causal_mask, pad_mask)
        x = x + self.dropout(attn_out)  # residual connection
        
        # Pre-norm FFN + residual
        ffn_out = self.ffn(self.norm2(x))
        x = x + ffn_out  # residual connection
        
        return x

print('DecoderLayer class defined OK')

DecoderLayer class defined OK


## Cell 7 — Full GPT Model

Stacks N DecoderLayers + final language model head (projects d_model → vocab_size logits)

In [7]:
class MedicalGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, ffn_dim,
                 max_seq_len, pad_id, dropout=0.1):
        super().__init__()
        self.pad_id = pad_id
        
        # Embeddings
        self.embeddings = Embeddings(vocab_size, d_model, max_seq_len, pad_id, dropout)
        
        # Stack of decoder layers
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_layers)
        ])
        
        # Final LayerNorm
        self.norm = nn.LayerNorm(d_model)
        
        # LM Head: d_model → vocab_size (one logit per vocab token)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying: share token embedding and lm_head weights
        # This is standard in GPT models — reduces parameters, improves performance
        self.lm_head.weight = self.embeddings.token_emb.weight
        
        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def make_causal_mask(self, seq_len, device):
        # Lower triangular matrix — token can only attend to itself and past
        # Example for seq_len=4:
        # [[1, 0, 0, 0],
        #  [1, 1, 0, 0],
        #  [1, 1, 1, 0],
        #  [1, 1, 1, 1]]
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
        return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, T, T)

    def forward(self, input_ids, attention_mask=None):
        # input_ids      : (batch, seq_len)
        # attention_mask : (batch, seq_len) — 1 for real, 0 for pad
        
        B, T = input_ids.shape
        device = input_ids.device
        
        # Embeddings
        x = self.embeddings(input_ids)  # (B, T, d_model)
        
        # Causal mask
        causal_mask = self.make_causal_mask(T, device)  # (1, 1, T, T)
        
        # Pass through all decoder layers
        for layer in self.layers:
            x = layer(x, causal_mask=causal_mask, pad_mask=attention_mask)
        
        # Final norm
        x = self.norm(x)  # (B, T, d_model)
        
        # LM head — project to vocab
        logits = self.lm_head(x)  # (B, T, vocab_size)
        
        return logits

print('MedicalGPT class defined OK')

MedicalGPT class defined OK


## Cell 8 — Instantiate Model and Count Parameters

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = MedicalGPT(
    vocab_size  = VOCAB_SIZE,
    d_model     = D_MODEL,
    n_heads     = N_HEADS,
    n_layers    = N_LAYERS,
    ffn_dim     = FFN_DIM,
    max_seq_len = MAX_SEQ_LEN,
    pad_id      = PAD_ID,
    dropout     = 0.1
).to(device)

# Count parameters
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\n=== Model Summary ===')
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')
print(f'Model size (approx)  : {total_params * 4 / 1024**2:.1f} MB')

print(f'\n=== Architecture ===')
print(f'Embedding layer      : {VOCAB_SIZE} x {D_MODEL}')
print(f'Decoder layers       : {N_LAYERS}')
print(f'Attention heads      : {N_HEADS} heads x {D_MODEL // N_HEADS} dim each')
print(f'FFN dimensions       : {D_MODEL} → {FFN_DIM} → {D_MODEL}')
print(f'LM Head              : {D_MODEL} → {VOCAB_SIZE} (weight tied)')

Using device: cuda

=== Model Summary ===
Total parameters     : 6,918,144
Trainable parameters : 6,918,144
Model size (approx)  : 26.4 MB

=== Architecture ===
Embedding layer      : 8000 x 256
Decoder layers       : 6
Attention heads      : 4 heads x 64 dim each
FFN dimensions       : 256 → 1024 → 256
LM Head              : 256 → 8000 (weight tied)
